# v20 GRPO Training Policy

Self-contained Kaggle/Colab notebook for the v20 constrained GRPO policy. Configure `HF_TOKEN` as a Kaggle Secret or environment variable before running.

In [ ]:
import os


def _read_kaggle_secret(names):
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as exc:
        print(f'Kaggle Secrets unavailable: {exc.__class__.__name__}')
        return ''

    client = UserSecretsClient()
    for name in names:
        try:
            value = (client.get_secret(name) or '').strip()
        except Exception as exc:
            print(f'Kaggle Secret {name} unavailable: {exc.__class__.__name__}')
            value = ''
        if value:
            return value
    return ''


def _resolve_hf_token():
    token = (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN') or '').strip()
    if not token:
        token = _read_kaggle_secret(('HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'))
    if not token:
        try:
            from getpass import getpass
            token = getpass('HF_TOKEN: ').strip()
        except Exception as exc:
            raise RuntimeError(
                'HF_TOKEN is required. On Kaggle, add a notebook Secret named HF_TOKEN '
                'or HUGGINGFACE_HUB_TOKEN and enable access for this notebook. '
                'Locally, set the HF_TOKEN environment variable before running.'
            ) from exc
    if not token:
        raise RuntimeError('HF_TOKEN is required and was empty.')
    os.environ['HF_TOKEN'] = token
    os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', token)


_resolve_hf_token()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
!pip install -q torch huggingface_hub matplotlib

In [ ]:
import os

defaults = {
    'CANDIDATES_CSV': '',
    'v20_GRPO_DATA_REMOTE_PATH': 'data/20260520_061012/candidates_v19.csv',
    'v20_SFT_REMOTE_PATH': 'v20/sft/model_weights_v20_sft.json',
    'v20_GRPO_EPOCHS': '60',
    'v20_GRPO_BATCH_GROUPS': '192',
    'v20_GRPO_SAMPLES': '12',
    'v20_GRPO_TEMPERATURE': '0.95',
    'v20_GRPO_LR': '0.00010',
    'v20_GRPO_WEIGHT_DECAY': '0.00018',
    'v20_GRPO_KL_WEIGHT': '0.35',
    'v20_GRPO_ENTROPY_WEIGHT': '0.030',
    'v20_GRPO_SUPERVISED_ANCHOR': '0.32',
    'v20_GRPO_PATIENCE': '14',
    'v20_GRPO_CHECKPOINT_EVERY': '30',
    'v20_GRPO_TOP1_DROP_TOLERANCE': '0.008',
    'v20_GRPO_MAX_KL': '0.42',
    'v20_GRPO_BLEND_CAP': '0.24',
    'v20_GRPO_CARRY_SFT_MEMBERS': '2',
    'v20_DEVICE': 'cuda',
}
for key, value in defaults.items():
    os.environ.setdefault(key, value)

print('GRPO config:', {key: os.environ[key] for key in defaults})

In [ ]:
v20_GRPO_CODE = 'import argparse\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\n\nHF_REPO_ID = "devaanshpa/orbit-wars-agent"\nHF_REPO_TYPE = "model"\nSFT_REMOTE_PREFIX = "v20/sft"\nGRPO_REMOTE_PREFIX = "v20/grpo"\nPINNED_DATASET = "data/20260520_061012/candidates_v19.csv"\n\nMETADATA_COLS = {\n    "label",\n    "selected",\n    "outcome_weight",\n    "game_result",\n    "reward_margin",\n    "agent_reward",\n    "opponent_reward",\n    "selected_heuristic_rank",\n    "counterfactual_positive",\n    "counterfactual_reason",\n    "failure_overcommit",\n    "failure_missed_tactical",\n    "failure_missed_comet",\n    "failure_slow_expansion",\n    "turn_advantage",\n    "future_advantage_delta_5",\n    "future_advantage_delta_15",\n    "future_advantage_delta_30",\n    "future_production_delta_15",\n    "future_planet_delta_15",\n    "game_id",\n    "candidate_id",\n    "version",\n    "cf_margin_delta",\n    "cf_prod_delta",\n    "cf_planet_delta",\n    "cf_survival",\n    "cf_crash",\n}\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(\n        description="Run v20 constrained GRPO-style policy improvement over Orbit Wars candidate groups."\n    )\n    parser.add_argument("--csv", default=os.environ.get("CANDIDATES_CSV", ""))\n    parser.add_argument(\n        "--data-remote-path",\n        default=os.environ.get("V20_GRPO_DATA_REMOTE_PATH", os.environ.get("v20_GRPO_DATA_REMOTE_PATH", PINNED_DATASET)),\n        help="Optional exact Hugging Face repo path for candidates_v20.csv or candidates_v19.csv. Defaults to the pinned counterfactual dataset.",\n    )\n    parser.add_argument(\n        "--prefer-local-data",\n        action="store_true",\n        help="Use a local candidates_v20.csv or candidates_v19.csv before trying Hugging Face. Default is Hugging Face.",\n    )\n    parser.add_argument(\n        "--sft-artifact",\n        default=os.environ.get("V20_SFT_ARTIFACT", os.environ.get("v20_SFT_ARTIFACT", "")),\n        help="Optional local SFT JSON override. By default GRPO downloads the SFT artifact from Hugging Face.",\n    )\n    parser.add_argument(\n        "--sft-remote-path",\n        default=os.environ.get("V20_SFT_REMOTE_PATH", os.environ.get("v20_SFT_REMOTE_PATH", f"{SFT_REMOTE_PREFIX}/model_weights_v20_sft.json")),\n        help="Hugging Face repo path for the SFT artifact used by GRPO.",\n    )\n    parser.add_argument(\n        "--prefer-local-sft",\n        action="store_true",\n        help="Use notebooks/v20/exports/sft/model_weights_v20_sft.json before trying Hugging Face.",\n    )\n    parser.add_argument("--export-dir", default="notebooks/v20/exports/grpo")\n    parser.add_argument("--epochs", type=int, default=int(os.environ.get("V20_GRPO_EPOCHS", os.environ.get("v20_GRPO_EPOCHS", "60"))))\n    parser.add_argument("--batch-groups", type=int, default=int(os.environ.get("V20_GRPO_BATCH_GROUPS", os.environ.get("v20_GRPO_BATCH_GROUPS", "192"))))\n    parser.add_argument("--samples-per-group", type=int, default=int(os.environ.get("V20_GRPO_SAMPLES", os.environ.get("v20_GRPO_SAMPLES", "12"))))\n    parser.add_argument("--temperature", type=float, default=float(os.environ.get("V20_GRPO_TEMPERATURE", os.environ.get("v20_GRPO_TEMPERATURE", "0.95"))))\n    parser.add_argument("--lr", type=float, default=float(os.environ.get("V20_GRPO_LR", os.environ.get("v20_GRPO_LR", "0.00010"))))\n    parser.add_argument("--weight-decay", type=float, default=float(os.environ.get("V20_GRPO_WEIGHT_DECAY", os.environ.get("v20_GRPO_WEIGHT_DECAY", "0.00018"))))\n    parser.add_argument("--kl-weight", type=float, default=float(os.environ.get("V20_GRPO_KL_WEIGHT", os.environ.get("v20_GRPO_KL_WEIGHT", "0.35"))))\n    parser.add_argument("--entropy-weight", type=float, default=float(os.environ.get("V20_GRPO_ENTROPY_WEIGHT", os.environ.get("v20_GRPO_ENTROPY_WEIGHT", "0.030"))))\n    parser.add_argument("--supervised-anchor", type=float, default=float(os.environ.get("V20_GRPO_SUPERVISED_ANCHOR", os.environ.get("v20_GRPO_SUPERVISED_ANCHOR", "0.32"))))\n    parser.add_argument("--patience", type=int, default=int(os.environ.get("V20_GRPO_PATIENCE", os.environ.get("v20_GRPO_PATIENCE", "14"))))\n    parser.add_argument("--checkpoint-every", type=int, default=int(os.environ.get("V20_GRPO_CHECKPOINT_EVERY", os.environ.get("v20_GRPO_CHECKPOINT_EVERY", "30"))))\n    parser.add_argument("--top1-drop-tolerance", type=float, default=float(os.environ.get("V20_GRPO_TOP1_DROP_TOLERANCE", "0.008")))\n    parser.add_argument("--max-kl", type=float, default=float(os.environ.get("V20_GRPO_MAX_KL", "0.42")))\n    parser.add_argument("--blend-cap", type=float, default=float(os.environ.get("V20_GRPO_BLEND_CAP", "0.24")))\n    parser.add_argument("--carry-sft-members", type=int, default=int(os.environ.get("V20_GRPO_CARRY_SFT_MEMBERS", "2")))\n    parser.add_argument(\n        "--device",\n        choices=("cuda", "cpu", "auto"),\n        default=os.environ.get("V20_DEVICE", os.environ.get("v20_DEVICE", "cuda")),\n        help="Training device. Defaults to CUDA for Kaggle 2*T4 runs.",\n    )\n    parser.add_argument("--seed", type=int, default=1819)\n    parser.add_argument("--upload", action="store_true")\n    parser.add_argument("--hf-repo-id", default=HF_REPO_ID)\n    parser.add_argument("--hf-repo-type", default=HF_REPO_TYPE)\n    return parser.parse_args()\n\n\ndef load_dotenv(path=".env"):\n    env_path = Path(path)\n    if not env_path.exists():\n        return\n    for raw_line in env_path.read_text(encoding="utf-8").splitlines():\n        line = raw_line.strip()\n        if not line or line.startswith("#") or "=" not in line:\n            continue\n        key, value = line.split("=", 1)\n        key = key.strip()\n        value = value.strip().strip("\\"\'")\n        if key and key not in os.environ:\n            os.environ[key] = value\n\n\ndef download_training_csv(remote_path, repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE):\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise RuntimeError("HF_TOKEN is required to download candidates_v20.csv/candidates_v19.csv from Hugging Face.")\n    try:\n        from huggingface_hub import HfApi, hf_hub_download\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to download data: pip install huggingface_hub") from exc\n\n    if not remote_path:\n        api = HfApi(token=token)\n        files = api.list_repo_files(repo_id=repo_id, repo_type=repo_type)\n        remote_csvs = sorted(\n            [\n                name\n                for name in files\n                if name.startswith("data/") and (name.endswith("/candidates_v20.csv") or name.endswith("/candidates_v19.csv"))\n            ],\n            reverse=True,\n        )\n        if not remote_csvs:\n            raise FileNotFoundError("No data/*/candidates_v20.csv or candidates_v19.csv found in Hugging Face repo.")\n        remote_path = remote_csvs[0]\n\n    return Path(\n        hf_hub_download(\n            repo_id=repo_id,\n            repo_type=repo_type,\n            filename=remote_path,\n            token=token,\n        )\n    )\n\n\ndef find_training_csv(csv_arg, remote_path="", prefer_local=False, repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE):\n    if csv_arg:\n        path = Path(csv_arg)\n        if not path.exists():\n            raise FileNotFoundError(f"Training CSV does not exist: {path}")\n        return path\n\n    if not prefer_local:\n        return download_training_csv(remote_path, repo_id=repo_id, repo_type=repo_type)\n\n    root = Path("data")\n    candidates = []\n    if root.exists():\n        candidates.extend(root.glob("*/candidates_v20.csv"))\n        candidates.extend(root.glob("*/candidates_v19.csv"))\n    if candidates:\n        return sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)[0]\n\n    return download_training_csv(remote_path, repo_id=repo_id, repo_type=repo_type)\n\n\ndef download_sft_artifact(remote_path, repo_id, repo_type):\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise RuntimeError("HF_TOKEN is required to download the SFT artifact from Hugging Face.")\n    try:\n        from huggingface_hub import hf_hub_download\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to download SFT artifacts.") from exc\n    return Path(\n        hf_hub_download(\n            repo_id=repo_id,\n            repo_type=repo_type,\n            filename=remote_path,\n            token=token,\n        )\n    )\n\n\ndef find_sft_artifact(path_arg, repo_id, repo_type, remote_path, prefer_local=False):\n    if path_arg:\n        path = Path(path_arg)\n        if not path.exists():\n            raise FileNotFoundError(f"SFT artifact does not exist: {path}")\n        return path\n\n    local = Path("notebooks/v20/exports/sft/model_weights_v20_sft.json")\n    if prefer_local and local.exists():\n        return local\n\n    return download_sft_artifact(remote_path, repo_id, repo_type)\n\n\ndef row_float(row, key, default=0.0):\n    try:\n        return float(row.get(key, default) or default)\n    except (TypeError, ValueError):\n        return float(default)\n\n\ndef read_rows(path):\n    with Path(path).open(newline="", encoding="utf-8") as f:\n        rows = list(csv.DictReader(f))\n    if not rows:\n        raise RuntimeError("Training CSV has no rows.")\n    feature_names = [name for name in rows[0] if name not in METADATA_COLS]\n    x_raw = [[row_float(row, name, 0.0) for name in feature_names] for row in rows]\n    labels = [max(0.0, min(1.0, row_float(row, "label", 0.0))) for row in rows]\n    selected = [row_float(row, "selected", 0.0) >= 0.5 for row in rows]\n    counterfactual = [row_float(row, "counterfactual_positive", 0.0) >= 0.5 for row in rows]\n    return rows, feature_names, x_raw, labels, selected, counterfactual\n\n\ndef split_by_game(rows, seed, validation_fraction=0.2):\n    games = sorted({row.get("game_id", "") for row in rows})\n    rng = random.Random(seed)\n    rng.shuffle(games)\n    valid_game_count = max(1, int(len(games) * validation_fraction)) if len(games) > 1 else 1\n    valid_games = set(games[:valid_game_count])\n    valid_indices = [i for i, row in enumerate(rows) if row.get("game_id", "") in valid_games]\n    valid_set = set(valid_indices)\n    train_indices = [i for i in range(len(rows)) if i not in valid_set] or valid_indices[:]\n    return train_indices, valid_indices, games, valid_games\n\n\ndef normalize_with_artifact(x_raw, feature_names, artifact):\n    member = artifact["members"][0] if artifact.get("members") else artifact\n    means = member.get("mean", {})\n    scales = member.get("scale", {})\n    normalized = []\n    for row in x_raw:\n        values = []\n        for name, value in zip(feature_names, row):\n            scale = float(scales.get(name, 1.0) or 1.0)\n            values.append((value - float(means.get(name, 0.0))) / scale)\n        normalized.append(values)\n    return normalized, means, scales\n\n\ndef build_groups(rows, indices):\n    grouped = defaultdict(list)\n    for raw_index in indices:\n        step = int(row_float(rows[raw_index], "step", 0.0))\n        grouped[(rows[raw_index].get("game_id", ""), step)].append(raw_index)\n    return [items for items in grouped.values() if len(items) >= 2]\n\n\ndef clamp(value, low, high):\n    return max(low, min(high, value))\n\n\ndef failure_exposure(row):\n    return (\n        row_float(row, "failure_overcommit", 0.0) * 1.25\n        + row_float(row, "failure_missed_tactical", 0.0) * 0.45\n        + row_float(row, "failure_missed_comet", 0.0) * 0.45\n        + row_float(row, "failure_slow_expansion", 0.0) * 0.35\n        + row_float(row, "kind_crash", 0.0) * 1.50\n        + max(0.0, row_float(row, "ship_cost_fraction", 0.0) - 0.62) * 0.55\n    )\n\n\ndef candidate_reward_components(row, label, selected, counterfactual):\n    """Reward-resistant offline proxy.\n\n    This reward relies heavily on causal counterfactual rollout deltas.\n    The selected bonus is kept small.\n    """\n    components = {\n        "label_signal": (label - 0.5) * 0.55,\n        "advantage_delta_15": clamp(row_float(row, "future_advantage_delta_15", 0.0) / 85.0, -1.25, 1.25) * 0.45,\n        "advantage_delta_30": clamp(row_float(row, "future_advantage_delta_30", 0.0) / 140.0, -0.90, 0.90) * 0.25,\n        "production_delta": clamp(row_float(row, "future_production_delta_15", 0.0) / 9.0, -0.75, 0.75) * 0.25,\n        "planet_delta": clamp(row_float(row, "future_planet_delta_15", 0.0) / 3.0, -0.75, 0.75) * 0.25,\n        "final_result": clamp(row_float(row, "game_result", 0.0), -1.0, 1.0) * 0.16,\n        "final_margin": clamp(row_float(row, "reward_margin", 0.0) / 800.0, -1.0, 1.0) * 0.12,\n        "selected_anchor": 0.05 if selected else 0.0,\n        "counterfactual_bonus": 0.22 if counterfactual else 0.0,\n\n        "cf_margin": clamp(row_float(row, "cf_margin_delta", 0.0) / 85.0, -1.5, 1.5) * 1.5,\n        "cf_prod": clamp(row_float(row, "cf_prod_delta", 0.0) / 9.0, -1.0, 1.0) * 0.75,\n        "cf_planet": clamp(row_float(row, "cf_planet_delta", 0.0) / 3.0, -1.0, 1.0) * 0.5,\n        "cf_survival_bonus": (row_float(row, "cf_survival", 1.0) - 1.0) * 2.0,\n        "cf_crash_penalty": -row_float(row, "cf_crash", 0.0) * 2.0,\n\n        "overcommit_penalty": -row_float(row, "failure_overcommit", 0.0) * 1.15,\n        "missed_tactical_penalty": -row_float(row, "failure_missed_tactical", 0.0) * 0.45,\n        "missed_comet_penalty": -row_float(row, "failure_missed_comet", 0.0) * 0.45,\n        "slow_expansion_penalty": -row_float(row, "failure_slow_expansion", 0.0) * 0.35,\n        "crash_penalty": -row_float(row, "kind_crash", 0.0) * 1.60,\n        "cost_penalty": -max(0.0, row_float(row, "ship_cost_fraction", 0.0) - 0.62) * 0.65,\n        "eta_penalty": -max(0.0, row_float(row, "eta_fraction_remaining", 0.0) - 0.45) * 0.22,\n    }\n    if row_float(row, "kind_defend", 0.0) >= 0.5 and row_float(row, "enemy_pressure", 0.0) > 0.0:\n        components["pressure_defense_bonus"] = clamp(row_float(row, "enemy_pressure", 0.0) / 65.0, 0.0, 0.35)\n    return components\n\n\ndef candidate_reward(row, label, selected, counterfactual):\n    return clamp(sum(candidate_reward_components(row, label, selected, counterfactual).values()), -4.0, 4.0)\n\n\ndef target_distribution(rows, rewards, group):\n    values = [math.exp(max(-6.0, min(6.0, rewards[i]))) for i in group]\n    total = sum(values)\n    return [value / total for value in values] if total > 0.0 else [1.0 / len(group)] * len(group)\n\n\ndef grouped_metrics(rows, scores, positive_mask, indices, rewards=None):\n    groups = build_groups(rows, indices)\n    hits = 0\n    total = 0\n    ranks = []\n    reward_gaps = []\n    top_rewards = []\n    failure_values = []\n    for group in groups:\n        positives = [i for i in group if positive_mask[i]]\n        if not positives:\n            continue\n        ordered = sorted(group, key=lambda i: scores[i], reverse=True)\n        total += 1\n        if positive_mask[ordered[0]]:\n            hits += 1\n        ranks.append(min(ordered.index(i) + 1 for i in positives) / max(1, len(ordered)))\n        top = ordered[0]\n        failure_values.append(failure_exposure(rows[top]))\n        if rewards is not None:\n            ordered_rewards = sorted(rewards[i] for i in group)\n            median_reward = ordered_rewards[len(ordered_rewards) // 2]\n            top_rewards.append(rewards[top])\n            reward_gaps.append(rewards[top] - median_reward)\n    return {\n        "top1": hits / total if total else 0.0,\n        "rank_fraction": sum(ranks) / len(ranks) if ranks else 1.0,\n        "turns": total,\n        "top_reward": sum(top_rewards) / len(top_rewards) if top_rewards else 0.0,\n        "reward_gap": sum(reward_gaps) / len(reward_gaps) if reward_gaps else 0.0,\n        "failure_exposure": sum(failure_values) / len(failure_values) if failure_values else 0.0,\n    }\n\n\ndef tune_blend(rows, probabilities, positive_mask, indices, score_scale, blend_cap):\n    """Pick a conservative model blend by validation top1, bounded by blend_cap."""\n    best = {"blend": 0.0, "top1": -1.0, "rank": 1.0}\n    heuristic_scores = [row_float(row, "heuristic_score_scaled", 0.0) * 100.0 for row in rows]\n    model_scores = [(prob - 0.5) * score_scale for prob in probabilities]\n    steps = max(1, int(round(max(0.0, blend_cap) * 100.0)))\n    for step in range(0, steps + 1):\n        blend = step / 100.0\n        mixed = [\n            heuristic_scores[i] * (1.0 - blend) + (heuristic_scores[i] + model_scores[i]) * blend\n            for i in range(len(rows))\n        ]\n        metrics = grouped_metrics(rows, mixed, positive_mask, indices)\n        if metrics["top1"] > best["top1"] or (\n            abs(metrics["top1"] - best["top1"]) <= 1e-9 and metrics["rank_fraction"] < best["rank"]\n        ):\n            best = {"blend": blend, "top1": metrics["top1"], "rank": metrics["rank_fraction"]}\n    return best\n\n\ndef make_model_class(torch, nn):\n    class CandidatePolicy(nn.Module):\n        def __init__(self, feature_count):\n            super().__init__()\n            self.net = nn.Sequential(\n                nn.Linear(feature_count, 128),\n                nn.ReLU(),\n                nn.Linear(128, 64),\n                nn.ReLU(),\n                nn.Linear(64, 32),\n                nn.ReLU(),\n                nn.Linear(32, 1),\n            )\n\n        def forward(self, inputs):\n            return self.net(inputs).view(-1)\n\n    return CandidatePolicy\n\n\ndef load_member_into_model(torch, model, member):\n    linear_layers = [module for module in model.net if module.__class__.__name__ == "Linear"]\n    for layer_module, layer_data in zip(linear_layers, member.get("layers", [])):\n        layer_module.weight.data = torch.tensor(layer_data["weights"], dtype=layer_module.weight.dtype, device=layer_module.weight.device)\n        layer_module.bias.data = torch.tensor(layer_data["bias"], dtype=layer_module.bias.dtype, device=layer_module.bias.device)\n\n\ndef choose_device(torch, args):\n    requested = args.device\n    if requested == "auto":\n        requested = "cuda" if torch.cuda.is_available() else "cpu"\n    if requested == "cuda" and torch.cuda.is_available():\n        return torch.device("cuda"), None\n    if requested == "cuda":\n        print("CUDA requested but unavailable; falling back to CPU.", flush=True)\n    return torch.device("cpu"), None\n\n\ndef optimizer_step(optimizer, xm):\n    if xm is None:\n        optimizer.step()\n    else:\n        xm.optimizer_step(optimizer, barrier=False)\n        xm.mark_step()\n\n\ndef layers_from_model(torch, nn, model):\n    linear_layers = [module for module in model.net if isinstance(module, nn.Linear)]\n    layers = []\n    for index, layer in enumerate(linear_layers):\n        layers.append(\n            {\n                "weights": layer.weight.detach().cpu().tolist(),\n                "bias": layer.bias.detach().cpu().tolist(),\n                "activation": "relu" if index < len(linear_layers) - 1 else "linear",\n            }\n        )\n    return layers\n\n\ndef maybe_upload_file(args, path, path_in_repo, commit_message):\n    if not args.upload:\n        return\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise RuntimeError("HF_TOKEN is required for checkpoint upload.")\n    try:\n        from huggingface_hub import HfApi\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to upload checkpoints: pip install huggingface_hub") from exc\n    api = HfApi(token=token)\n    api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n    api.upload_file(\n        path_or_fileobj=str(path),\n        path_in_repo=path_in_repo,\n        repo_id=args.hf_repo_id,\n        repo_type=args.hf_repo_type,\n        commit_message=commit_message,\n    )\n    print(f"Uploaded checkpoint to https://huggingface.co/{args.hf_repo_id}/blob/main/{path_in_repo}", flush=True)\n\n\ndef save_grpo_checkpoint(args, torch, nn, model, epoch, item, history, feature_names, means, scales, sft_path, blend):\n    checkpoint_dir = Path(args.export_dir) / "checkpoints"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    member = {\n        "version": "v20_grpo",\n        "model_type": "mlp_relu_candidate_ranker",\n        "features": feature_names,\n        "mean": means,\n        "scale": scales,\n        "layers": layers_from_model(torch, nn, model),\n        "activation": "relu",\n        "score_scale": 235.0,\n    }\n    payload = {\n        "version": "v20_grpo_checkpoint",\n        "created_at": int(time.time()),\n        "epoch": epoch,\n        "latest_metrics": item,\n        "history": history,\n        "source_sft_artifact": str(sft_path),\n        "model_type": "ensemble_mlp_relu_candidate_ranker",\n        "members": [member],\n        "blend": blend,\n    }\n    path = checkpoint_dir / f"grpo_epoch_{epoch:03d}.json"\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")\n    print(f"Saved v20 GRPO checkpoint: {path}", flush=True)\n    maybe_upload_file(\n        args,\n        path,\n        f"{GRPO_REMOTE_PREFIX}/checkpoints/{path.name}",\n        f"Upload v20 GRPO checkpoint epoch {epoch}",\n    )\n\n\ndef sigmoid_prob(value):\n    value = max(-50.0, min(50.0, value))\n    return 1.0 / (1.0 + math.exp(-value))\n\n\ndef train(args):\n    try:\n        import torch\n        from torch import nn\n        import torch.nn.functional as functional\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("PyTorch is required for v20 GRPO training. Install torch or use Kaggle/Colab.") from exc\n\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    data_path = find_training_csv(\n        args.csv,\n        remote_path=args.data_remote_path,\n        prefer_local=args.prefer_local_data,\n        repo_id=args.hf_repo_id,\n        repo_type=args.hf_repo_type,\n    )\n    sft_path = find_sft_artifact(\n        args.sft_artifact,\n        args.hf_repo_id,\n        args.hf_repo_type,\n        args.sft_remote_path,\n        prefer_local=args.prefer_local_sft,\n    )\n    sft_artifact = json.loads(Path(sft_path).read_text(encoding="utf-8"))\n    rows, feature_names, x_raw, labels, selected, counterfactual = read_rows(data_path)\n    all_x, means, scales = normalize_with_artifact(x_raw, feature_names, sft_artifact)\n    train_indices, valid_indices, games, valid_games = split_by_game(rows, args.seed)\n    train_groups = build_groups(rows, train_indices)\n    valid_groups = build_groups(rows, valid_indices)\n    rewards = [\n        candidate_reward(row, labels[i], selected[i], counterfactual[i])\n        for i, row in enumerate(rows)\n    ]\n    positive_mask = [selected[i] or counterfactual[i] or labels[i] >= 0.55 for i in range(len(rows))]\n    device, xm = choose_device(torch, args)\n    CandidatePolicy = make_model_class(torch, nn)\n    base_members = sft_artifact.get("members", []) or [sft_artifact]\n    model = CandidatePolicy(len(feature_names)).to(device)\n    ref_model = CandidatePolicy(len(feature_names)).to(device)\n    load_member_into_model(torch, model, base_members[0])\n    load_member_into_model(torch, ref_model, base_members[0])\n    ref_model.eval()\n    for param in ref_model.parameters():\n        param.requires_grad_(False)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, args.epochs))\n    x_all = torch.tensor(all_x, dtype=torch.float32, device=device)\n\n    print(\n        json.dumps(\n            {\n                "csv": str(data_path),\n                "data_remote_path": args.data_remote_path or "latest data/*/candidates_v20.csv or candidates_v19.csv",\n                "data_source": "local_override" if args.csv else ("local_preferred" if args.prefer_local_data else "hugging_face"),\n                "sft_artifact": str(sft_path),\n                "sft_remote_path": args.sft_remote_path,\n                "sft_source": "local_override" if args.sft_artifact else ("local_preferred" if args.prefer_local_sft else "hugging_face"),\n                "rows": len(rows),\n                "features": len(feature_names),\n                "games": len(games),\n                "validation_games": len(valid_games),\n                "train_groups": len(train_groups),\n                "validation_groups": len(valid_groups),\n                "samples_per_group": args.samples_per_group,\n                "device": str(device),\n                "checkpoint_every": args.checkpoint_every,\n                "top1_drop_tolerance": args.top1_drop_tolerance,\n                "max_kl": args.max_kl,\n                "blend_cap": args.blend_cap,\n                "carry_sft_members": args.carry_sft_members,\n                "remote_upload_path": GRPO_REMOTE_PREFIX,\n            },\n            indent=2,\n        ),\n        flush=True,\n    )\n\n    model.eval()\n    with torch.no_grad():\n        ref_logits_all = ref_model(x_all).detach().cpu().tolist()\n    ref_scores = [sigmoid_prob(value) for value in ref_logits_all]\n    baseline_valid_metrics = grouped_metrics(rows, ref_scores, positive_mask, valid_indices, rewards)\n    baseline_train_metrics = grouped_metrics(rows, ref_scores, positive_mask, train_indices, rewards)\n    baseline_objective = (\n        (1.0 - baseline_valid_metrics["top1"])\n        + 0.35 * baseline_valid_metrics["rank_fraction"]\n        + baseline_valid_metrics["failure_exposure"] * 0.16\n    )\n    best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n    best_objective = baseline_objective\n    best_source = "sft_baseline"\n    stale = 0\n    history = []\n    safety_violations = 0\n    checkpoint_blend = max(0.0, min(args.blend_cap, float(sft_artifact.get("blend", 0.18))))\n    print(\n        "grpo baseline "\n        f"valid_top1={baseline_valid_metrics[\'top1\']:.4f} "\n        f"rank={baseline_valid_metrics[\'rank_fraction\']:.4f} "\n        f"reward_gap={baseline_valid_metrics[\'reward_gap\']:.4f} "\n        f"fail={baseline_valid_metrics[\'failure_exposure\']:.4f} "\n        f"objective={baseline_objective:.5f}",\n        flush=True,\n    )\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        groups = train_groups[:]\n        random.shuffle(groups)\n        total_loss = 0.0\n        total_policy = 0.0\n        total_kl = 0.0\n        total_entropy = 0.0\n        batches = 0\n        for offset in range(0, len(groups), args.batch_groups):\n            batch_groups = groups[offset : offset + args.batch_groups]\n            optimizer.zero_grad(set_to_none=True)\n            loss_acc = None\n            policy_acc = 0.0\n            kl_acc = 0.0\n            entropy_acc = 0.0\n            for group in batch_groups:\n                indices = torch.tensor(group, dtype=torch.long, device=device)\n                logits = model(x_all[indices]) / max(0.20, args.temperature)\n                with torch.no_grad():\n                    ref_logits = ref_model(x_all[indices]) / max(0.20, args.temperature)\n                probs = torch.softmax(logits, dim=0)\n                ref_probs = torch.softmax(ref_logits, dim=0)\n                log_probs = torch.log(probs.clamp_min(1e-8))\n                group_rewards = torch.tensor([rewards[i] for i in group], dtype=torch.float32, device=device)\n                sampled = torch.multinomial(probs.detach(), num_samples=min(args.samples_per_group, len(group)), replacement=True)\n                sampled_rewards = group_rewards[sampled]\n                if sampled_rewards.numel() > 1:\n                    advantages = (sampled_rewards - sampled_rewards.mean()) / sampled_rewards.std(unbiased=False).clamp_min(1e-4)\n                else:\n                    advantages = sampled_rewards - sampled_rewards.mean()\n                policy_loss = -(advantages.detach() * log_probs[sampled]).mean()\n                kl = (probs * (torch.log(probs.clamp_min(1e-8)) - torch.log(ref_probs.clamp_min(1e-8)))).sum()\n                entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum()\n                target = torch.tensor(target_distribution(rows, rewards, group), dtype=torch.float32, device=device)\n                anchor = -(target * functional.log_softmax(logits, dim=0)).sum()\n                loss = policy_loss + args.kl_weight * kl - args.entropy_weight * entropy + args.supervised_anchor * anchor\n                loss_acc = loss if loss_acc is None else loss_acc + loss\n                policy_acc += float(policy_loss.detach().cpu())\n                kl_acc += float(kl.detach().cpu())\n                entropy_acc += float(entropy.detach().cpu())\n            if loss_acc is None:\n                continue\n            loss_acc = loss_acc / max(1, len(batch_groups))\n            loss_acc.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.8)\n            optimizer_step(optimizer, xm)\n            total_loss += float(loss_acc.detach().cpu())\n            total_policy += policy_acc / max(1, len(batch_groups))\n            total_kl += kl_acc / max(1, len(batch_groups))\n            total_entropy += entropy_acc / max(1, len(batch_groups))\n            batches += 1\n        scheduler.step()\n\n        if epoch >= 1:\n            model.eval()\n            with torch.no_grad():\n                logits_all = model(x_all).detach().cpu().tolist()\n            scores = [sigmoid_prob(value) for value in logits_all]\n            valid_metrics = grouped_metrics(rows, scores, positive_mask, valid_indices, rewards)\n            avg_kl = total_kl / max(1, batches)\n            top1_drop = baseline_valid_metrics["top1"] - valid_metrics["top1"]\n            kl_excess = max(0.0, avg_kl - args.max_kl)\n            top1_excess = max(0.0, top1_drop - args.top1_drop_tolerance)\n            reward_hack_risk = (\n                valid_metrics["failure_exposure"] * 0.16\n                + max(0.0, -valid_metrics["reward_gap"]) * 0.06\n                + kl_excess * 0.18\n            )\n            objective = (\n                (1.0 - valid_metrics["top1"])\n                + 0.35 * valid_metrics["rank_fraction"]\n                + 0.06 * avg_kl\n                + reward_hack_risk\n                + 10.0 * top1_excess\n                + 0.50 * kl_excess\n            )\n            safe_to_accept = top1_drop <= args.top1_drop_tolerance and avg_kl <= args.max_kl\n            item = {\n                "epoch": epoch,\n                "loss": total_loss / max(1, batches),\n                "policy_loss": total_policy / max(1, batches),\n                "kl": avg_kl,\n                "entropy": total_entropy / max(1, batches),\n                "valid_turn_top1": valid_metrics["top1"],\n                "valid_rank_fraction": valid_metrics["rank_fraction"],\n                "valid_reward_gap": valid_metrics["reward_gap"],\n                "valid_top_reward": valid_metrics["top_reward"],\n                "valid_failure_exposure": valid_metrics["failure_exposure"],\n                "baseline_valid_top1": baseline_valid_metrics["top1"],\n                "top1_drop_from_baseline": top1_drop,\n                "safe_to_accept": safe_to_accept,\n                "reward_hack_risk": reward_hack_risk,\n                "objective": objective,\n                "lr": scheduler.get_last_lr()[0],\n            }\n            history.append(item)\n            print(\n                f"grpo epoch={epoch:03d} loss={item[\'loss\']:.5f} "\n                f"policy={item[\'policy_loss\']:.5f} kl={item[\'kl\']:.5f} entropy={item[\'entropy\']:.5f} "\n                f"valid_top1={item[\'valid_turn_top1\']:.4f} rank={item[\'valid_rank_fraction\']:.4f} "\n                f"reward_gap={item[\'valid_reward_gap\']:.4f} fail={item[\'valid_failure_exposure\']:.4f} "\n                f"drop={item[\'top1_drop_from_baseline\']:.4f} safe={int(safe_to_accept)} "\n                f"hack_risk={item[\'reward_hack_risk\']:.4f}",\n                flush=True,\n            )\n            if args.checkpoint_every > 0 and epoch % args.checkpoint_every == 0:\n                save_grpo_checkpoint(\n                    args,\n                    torch,\n                    nn,\n                    model,\n                    epoch,\n                    item,\n                    history,\n                    feature_names,\n                    means,\n                    scales,\n                    sft_path,\n                    checkpoint_blend,\n                )\n            if safe_to_accept and objective + 1e-5 < best_objective:\n                best_objective = objective\n                best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n                best_source = f"epoch_{epoch:03d}"\n                stale = 0\n            else:\n                if not safe_to_accept:\n                    safety_violations += 1\n                stale += 1\n                if stale >= args.patience:\n                    print(\n                        f"grpo early_stop epoch={epoch} best_objective={best_objective:.5f} "\n                        f"best_source={best_source} safety_violations={safety_violations}",\n                        flush=True,\n                    )\n                    break\n\n    if best_state is not None:\n        model.load_state_dict(best_state)\n    model.eval()\n    with torch.no_grad():\n        logits_all = model(x_all).detach().cpu().tolist()\n    all_probs = [sigmoid_prob(value) for value in logits_all]\n    train_metrics = grouped_metrics(rows, all_probs, positive_mask, train_indices, rewards)\n    valid_metrics = grouped_metrics(rows, all_probs, positive_mask, valid_indices, rewards)\n\n    score_scale = 235.0\n    tuned_blend = tune_blend(rows, all_probs, positive_mask, valid_indices, score_scale, args.blend_cap)\n    blend = max(0.0, min(args.blend_cap, tuned_blend["blend"]))\n    member = {\n        "version": "v20_grpo",\n        "model_type": "mlp_relu_candidate_ranker",\n        "features": feature_names,\n        "mean": means,\n        "scale": scales,\n        "layers": layers_from_model(torch, nn, model),\n        "activation": "relu",\n        "score_scale": score_scale,\n    }\n    carried_members = []\n    for base_member in base_members[: max(0, args.carry_sft_members)]:\n        if base_member:\n            carried_members.append(base_member)\n    members = carried_members + [member]\n    metrics = {\n        "rows": len(rows),\n        "features": len(feature_names),\n        "games": len(games),\n        "train_groups": len(train_groups),\n        "validation_groups": len(valid_groups),\n        "baseline_train_turn_top1_positive_rate": baseline_train_metrics["top1"],\n        "baseline_validation_turn_top1_positive_rate": baseline_valid_metrics["top1"],\n        "baseline_validation_positive_mean_rank_fraction": baseline_valid_metrics["rank_fraction"],\n        "baseline_validation_reward_gap": baseline_valid_metrics["reward_gap"],\n        "train_turn_top1_positive_rate": train_metrics["top1"],\n        "validation_turn_top1_positive_rate": valid_metrics["top1"],\n        "train_positive_mean_rank_fraction": train_metrics["rank_fraction"],\n        "validation_positive_mean_rank_fraction": valid_metrics["rank_fraction"],\n        "train_reward_gap": train_metrics["reward_gap"],\n        "validation_reward_gap": valid_metrics["reward_gap"],\n        "train_top_reward": train_metrics["top_reward"],\n        "validation_top_reward": valid_metrics["top_reward"],\n        "train_failure_exposure": train_metrics["failure_exposure"],\n        "validation_failure_exposure": valid_metrics["failure_exposure"],\n        "blend": blend,\n        "blend_cap": args.blend_cap,\n        "tuned_blend_validation_top1": tuned_blend["top1"],\n        "tuned_blend_validation_rank_fraction": tuned_blend["rank"],\n        "score_scale": score_scale,\n        "device": str(device),\n        "kl_weight": args.kl_weight,\n        "entropy_weight": args.entropy_weight,\n        "supervised_anchor": args.supervised_anchor,\n        "top1_drop_tolerance": args.top1_drop_tolerance,\n        "max_kl": args.max_kl,\n        "best_source": best_source,\n        "safety_violations": safety_violations,\n        "carried_sft_members": len(carried_members),\n        "exported_members": len(members),\n        "reward_design": "bounded_counterfactual_reward_with_sft_kl_top1_gate_and_blend_cap",\n    }\n    artifact = {\n        "version": "v20_grpo",\n        "created_at": int(time.time()),\n        "source_csv": str(data_path),\n        "source_sft_artifact": str(sft_path),\n        "model_type": "ensemble_mlp_relu_candidate_ranker",\n        "members": members,\n        "blend": blend,\n        "metrics": metrics,\n    }\n\n    export_dir = Path(args.export_dir)\n    graph_dir = export_dir / "graphs"\n    export_dir.mkdir(parents=True, exist_ok=True)\n    graph_dir.mkdir(parents=True, exist_ok=True)\n    (export_dir / "model_weights_v20_grpo.json").write_text(json.dumps(artifact, indent=2, sort_keys=True), encoding="utf-8")\n    (export_dir / "metrics_v20_grpo.json").write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8")\n    (export_dir / "training_history_v20_grpo.json").write_text(json.dumps(history, indent=2, sort_keys=True), encoding="utf-8")\n    with (export_dir / "predictions_v20_grpo.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.writer(f)\n        writer.writerow(["row_index", "reward", "failure_exposure", "label", "prediction", "selected", "counterfactual_positive", "split"])\n        valid_set = set(valid_indices)\n        for i, pred in enumerate(all_probs):\n            writer.writerow([\n                i,\n                rewards[i],\n                failure_exposure(rows[i]),\n                labels[i],\n                pred,\n                float(selected[i]),\n                float(counterfactual[i]),\n                "validation" if i in valid_set else "train",\n            ])\n\n    try:\n        import matplotlib.pyplot as plt\n\n        epochs = [item["epoch"] for item in history]\n        plt.figure(figsize=(7, 4))\n        plt.plot(epochs, [item["valid_turn_top1"] for item in history], label="validation top1")\n        plt.plot(epochs, [item["kl"] for item in history], label="KL")\n        plt.plot(epochs, [item["entropy"] for item in history], label="entropy")\n        plt.xlabel("epoch")\n        plt.title("v20 GRPO diagnostics")\n        plt.legend()\n        plt.tight_layout()\n        plt.savefig(graph_dir / "grpo_diagnostics_v20.png", dpi=150)\n        plt.close()\n    except ModuleNotFoundError:\n        print("matplotlib is not installed; skipped graph generation.", flush=True)\n\n    print(json.dumps(metrics, indent=2, sort_keys=True), flush=True)\n    print(f"Saved v20 GRPO artifact: {export_dir / \'model_weights_v20_grpo.json\'}", flush=True)\n\n    if args.upload:\n        load_dotenv()\n        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n        if not token:\n            raise RuntimeError("HF_TOKEN is required for --upload.")\n        try:\n            from huggingface_hub import HfApi\n        except ModuleNotFoundError as exc:\n            raise RuntimeError("Install huggingface_hub to upload: pip install huggingface_hub") from exc\n        api = HfApi(token=token)\n        api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n        api.upload_folder(\n            folder_path=str(export_dir),\n            repo_id=args.hf_repo_id,\n            repo_type=args.hf_repo_type,\n            path_in_repo=GRPO_REMOTE_PREFIX,\n            commit_message="Upload v20 GRPO Orbit Wars policy artifacts and graphs",\n        )\n        print(f"Uploaded {export_dir} to https://huggingface.co/{args.hf_repo_id}/tree/main/{GRPO_REMOTE_PREFIX}", flush=True)\n\n\ndef main():\n    train(parse_args())\n\n\nif __name__ == "__main__":\n    main()\n'
print(f'Loaded v20_grpo trainer: {len(v20_GRPO_CODE)} chars')

In [ ]:
import argparse
import os
from pathlib import Path

export_dir = Path('/kaggle/working/v20_exports/grpo') if Path('/kaggle/working').exists() else Path('notebooks/v20/exports/grpo')
args = argparse.Namespace(
    csv=os.environ.get('CANDIDATES_CSV', ''),
    data_remote_path=os.environ.get('v20_GRPO_DATA_REMOTE_PATH', 'data/20260520_061012/candidates_v19.csv'),
    prefer_local_data=False,
    sft_artifact=os.environ.get('v20_SFT_ARTIFACT', ''),
    sft_remote_path=os.environ.get('v20_SFT_REMOTE_PATH', 'v20/sft/model_weights_v20_sft.json'),
    prefer_local_sft=False,
    export_dir=str(export_dir),
    epochs=int(os.environ['v20_GRPO_EPOCHS']),
    batch_groups=int(os.environ['v20_GRPO_BATCH_GROUPS']),
    samples_per_group=int(os.environ['v20_GRPO_SAMPLES']),
    temperature=float(os.environ['v20_GRPO_TEMPERATURE']),
    lr=float(os.environ['v20_GRPO_LR']),
    weight_decay=float(os.environ['v20_GRPO_WEIGHT_DECAY']),
    kl_weight=float(os.environ['v20_GRPO_KL_WEIGHT']),
    entropy_weight=float(os.environ['v20_GRPO_ENTROPY_WEIGHT']),
    supervised_anchor=float(os.environ['v20_GRPO_SUPERVISED_ANCHOR']),
    patience=int(os.environ['v20_GRPO_PATIENCE']),
    checkpoint_every=int(os.environ['v20_GRPO_CHECKPOINT_EVERY']),
    top1_drop_tolerance=float(os.environ['v20_GRPO_TOP1_DROP_TOLERANCE']),
    max_kl=float(os.environ['v20_GRPO_MAX_KL']),
    blend_cap=float(os.environ['v20_GRPO_BLEND_CAP']),
    carry_sft_members=int(os.environ['v20_GRPO_CARRY_SFT_MEMBERS']),
    device=os.environ.get('v20_DEVICE', 'cuda'),
    seed=1819,
    upload=True,
    hf_repo_id='devaanshpa/orbit-wars-agent',
    hf_repo_type='model',
)
print('Running v20 GRPO with:', vars(args))
v20_grpo_ns = {}
exec(v20_GRPO_CODE, v20_grpo_ns)
v20_grpo_ns['train'](args)